Date Created: 21/08/2026

Last Edited: 21/08/2026

In [0]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report

from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [0]:
import ast

In [0]:
from sklearn.calibration import CalibratedClassifierCV

# Load the datasets

In [0]:
df = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_dataset_CLEAN.csv')

In [0]:
data = pd.read_csv('/Volumes/projects/default/exercise-analysis/programs_detailed_boostcamp_kaggle.csv')

In [0]:
summary = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-analysis-datasets/program_summary.csv')

In [0]:
data.head()

# Run Naiive Term Frequency Model

In [0]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

In [0]:
labeled = df
labeled['norm'] = labeled['exercise'].apply(norm)

prog = data
unique_names = pd.DataFrame({'exercise_name': prog['exercise_name'].unique()})
unique_names['norm'] = unique_names['exercise_name'].apply(norm)

# Layer 1: exact-match bootstrap

In [0]:
lookup = labeled.drop_duplicates('norm').set_index('norm')['label']

In [0]:
unique_names['label'] = unique_names['norm'].map(lookup)
to_predict = unique_names[unique_names['label'].isna()].copy()
# print(f"{unique_names['label'].notna().sum()} matched directly, {len(to_predict)} need prediction")

# Layer 2: Train Classifier on labeled set

In [0]:
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words="english")),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=1)),
])
pipe = Pipeline([("feats", vectorizer), ("clf", LinearSVC(class_weight="balanced", max_iter=5000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, labeled['exercise'], labeled['label'], cv=cv, scoring='f1_macro', n_jobs=-1)
print("CV macro-F1:", scores.mean(), scores.std())

pipe.fit(labeled['exercise'], labeled['label'])
to_predict['label'] = pipe.predict(to_predict['exercise_name'])

## Calibrated probabilities to estimate errors

In [0]:
calibrated = CalibratedClassifierCV(pipe, cv=5, method='sigmoid')
calibrated.fit(labeled['exercise'], labeled['label'])

probs = calibrated.predict_proba(to_predict['exercise_name'])
to_predict['confidence'] = probs.max(axis=1)
to_predict.sort_values('confidence').head(30)  # review these first

In [0]:
to_predict.shape

In [0]:
to_predict.confidence.plot.hist(bins=20)

In [0]:
to_predict.confidence.plot.kde()
plt.xlim(0,1)

In [0]:
to_predict['confidence'].plot(kind='hist', cumulative=True, density=True)

In [0]:
to_predict.to_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_to_predict.csv')

# Labelling completed --> Allocate to original dataset
(Including a manual revision of uncertain values)